# 파서학개론

## PDF란?

### **PDF 기본 요소**
**PDF의 가장 기본 요소 = 객체**

두 종류의 객체가 존재함.
1. 직접 객체
2. 간접 객체

**1. 직접 객체**
```
true false                       % Boolean
123 -98 34.5                     % Number
(Hello)                          % Literal string
<48656C6C6F>                     % Hexadecimal string
/Type /Catalog                   % Name object
[0 0 612 792]                    % Array
<< /Type /Page /Parent 2 0 R >>  % Dictionary
stream ... endstream             % Stream object
null                             % Null object
```

**2. 간접 객체**
```
12 0 obj ... endobj              % 간접 객체 선언
12 0 R                           % 간접 객체 호출
```

*\+ 간접 객체 식별 방법*
```
<object number> <generation number> ...
```

---

### **파일 구조**

#### **PDF = Header + Body + Cross-reference table + Trailer**

Header : 파일의 첫 줄. PDF 규격의 버전을 식별함.

Body : 간접 객체들 + 문서 요소 표현 + 객체 스트림

Cross-reference table : 간접 객체들이 파일의 어느 위치에 있는지(바이트 오프셋)에 대한 정보를 담고 있는 테이블.

Trailer : PDF 리더기가 문서의 구조를 파악하기 위해 가장 먼저 참고하는 영역.

---

#### PDF는 뒤에서 앞으로 해석!!!

---

### **[1] Trailer**

**Trailer 역할**
1. Cross-reference table의 위치 제공
2. Body의 특정 특수 객체들의 위치 제공
3. 파일의 끝을 알리는 %%EOF 마커
4. 상호 참조 테이블이 시작되는 바이트 오프셋 위치(startxref)
5. 문서 계층 구조의 최상위 객체인 카탈로그 딕셔너리(Root) 정보

**Trailer 예시**
```
trailer
<<
    /Size 117
    /Root 1 0 R
    /Info 14 0 R
    /ID[
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
    ] 
>>
startxref
27593
%%EOF
```

Trailer는 PDF 파일을 수정하면 추가될 수 있다. 그런 경우 추가된 객체와 /Prev 와 /XReStm 등으로 추가 정보를 표현한다.

예시
```
trailer
<<
    /Size 117
    /Root 1 0 R
    /Info 14 0 R
    /ID[
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
        <9B8EA23639C1DA4FA67B9B17AEFF023D>
    ] 
>>
startxref
27593
%%EOF

xref
0 0
trailer
<<
/Size 117
/Root 1 0 R
/Info 14 0 R
/ID[
    <9B8EA23639C1DA4FA67B9B17AEFF023D>
    <9B8EA23639C1DA4FA67B9B17AEFF023D>
] 
/Prev 27593
/XRefStm 27121
>>
startxref
30092
%%EOF
```

---

### **[2] Cross-reference table**

줄여서 xref 라고 PDF에 표기함.

trailer에 존재하는 startxref가 xref의 위치를 알려줌.

xref 태그부터 trailer 태그 전까지가 모두 Cross-reference table 영역.

예시
```
xref
0 6
0000000003 65535 f
0000000017 00000 n
0000000081 00000 n
0000000000 00007 f
0000000331 00000 n
0000000409 00000 n
```

의미 해석

```
xref = cross-reference table 시작
0 6 = 0부터 시작하는 객체가 6개 있다.
0000000003 65535 f = 0번 객체
0000000017 00000 n = 1번 객체
0000000081 00000 n = 2번 객체
0000000000 00007 f = 3번 객체
0000000331 00000 n = 4번 객체
0000000409 00000 n = 5번 객체
```

**객체 사용 여부**
```
n = in-use (사용 중) | f = free (사용 안함)
```



**앞의 두 숫자의 의미**
1. n인 경우
```
0000000017 00000 n

<byte offset> <generation> n
```

해석 >> 17 바이트 위치에 해당 객체가 시작되고, 이 객체의 세대(버전)는 0번 이고, 사용되고 있는 객체이다.

2. f인 경우
```
0000000003 65535 f

<next free obj num> <generation> f
```
해석 >> 다음 free 객체의 번호는 3 이고, 더 이상 재사용 금지! (65535는 PDF generation number의 최대값)

---

#### **(실습) PDF에서 Trailer와 Cross-reference table 확인해보기**

In [21]:
# PDF 선택
pdf_path = "파서학개론.pdf"

In [22]:
# PDF를 byte로 읽기
raw_pdf = None

with open(pdf_path, "rb") as f:
    raw_pdf = f.read()

In [23]:
# header 100 바이트만 출력
print(raw_pdf[:100])

b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(ko-KR) /StructTreeRoot 15 0 R/MarkInfo<</'


In [24]:
# tail 100 바이트만 출력
print(raw_pdf[-100:])

b'B17AEFF023D><9B8EA23639C1DA4FA67B9B17AEFF023D>] /Prev 27593/XRefStm 27121>>\r\nstartxref\r\n30092\r\n%%EOF'


In [25]:
# 예쁘게 보기 위해서 latin-1 디코딩 적용
print(raw_pdf[:100].decode('latin-1'))

%PDF-1.7
%µµµµ
1 0 obj
<</Type/Catalog/Pages 2 0 R/Lang(ko-KR) /StructTreeRoot 15 0 R/MarkInfo<</


In [26]:
print(raw_pdf[-100:].decode('latin-1'))

B17AEFF023D><9B8EA23639C1DA4FA67B9B17AEFF023D>] /Prev 27593/XRefStm 27121>>
startxref
30092
%%EOF


In [27]:
# trailer의 구성요소 중 하나인 %%EOF 확인 가능
print(raw_pdf[-5:].decode("latin-1"))

%%EOF


In [37]:
# startxref를 따라 30092 byteoffset에서 끝까지(xref + trailer) 출력
print(raw_pdf[30092:].decode("latin-1"))

xref
0 0
trailer
<</Size 117/Root 1 0 R/Info 14 0 R/ID[<9B8EA23639C1DA4FA67B9B17AEFF023D><9B8EA23639C1DA4FA67B9B17AEFF023D>] /Prev 27593/XRefStm 27121>>
startxref
30092
%%EOF


In [38]:
# 깔끔하게 보기 위해서 / 가 나오면 개행 진행.
xref_trailer1 = raw_pdf[30092:].decode("latin-1")
xref_trailer1 = xref_trailer1.replace('/', '\n/')
print(xref_trailer1)

xref
0 0
trailer
<<
/Size 117
/Root 1 0 R
/Info 14 0 R
/ID[<9B8EA23639C1DA4FA67B9B17AEFF023D><9B8EA23639C1DA4FA67B9B17AEFF023D>] 
/Prev 27593
/XRefStm 27121>>
startxref
30092
%%EOF


In [39]:
# /Prev를 통해 이전 xref + trailer 확인
xref_trailer2 = raw_pdf[27593:].decode("latin-1")
xref_trailer2 = xref_trailer2.replace('/', '\n/')
print(xref_trailer2)

xref
0 117
0000000015 65535 f
0000000017 00000 n
0000000168 00000 n
0000000224 00000 n
0000000508 00000 n
0000001848 00000 n
0000001984 00000 n
0000002012 00000 n
0000002175 00000 n
0000002248 00000 n
0000002494 00000 n
0000002548 00000 n
0000002602 00000 n
0000002777 00000 n
0000003024 00000 n
0000000016 65535 f
0000000017 65535 f
0000000018 65535 f
0000000019 65535 f
0000000020 65535 f
0000000021 65535 f
0000000022 65535 f
0000000023 65535 f
0000000024 65535 f
0000000025 65535 f
0000000026 65535 f
0000000027 65535 f
0000000028 65535 f
0000000029 65535 f
0000000030 65535 f
0000000031 65535 f
0000000032 65535 f
0000000033 65535 f
0000000034 65535 f
0000000035 65535 f
0000000036 65535 f
0000000037 65535 f
0000000038 65535 f
0000000039 65535 f
0000000040 65535 f
0000000041 65535 f
0000000042 65535 f
0000000043 65535 f
0000000044 65535 f
0000000045 65535 f
0000000046 65535 f
0000000047 65535 f
0000000048 65535 f
0000000049 65535 f
0000000050 65535 f
0000000051 65535 f
0000000052 65535 f
0

In [40]:
# 이전 xref + trailer만 출력
# /Prev를 통해 이전 xref + trailer 확인
xref_trailer2_only = raw_pdf[27593:30092].decode("latin-1")
xref_trailer2_only = xref_trailer2_only.replace('/', '\n/')
print(xref_trailer2_only)

xref
0 117
0000000015 65535 f
0000000017 00000 n
0000000168 00000 n
0000000224 00000 n
0000000508 00000 n
0000001848 00000 n
0000001984 00000 n
0000002012 00000 n
0000002175 00000 n
0000002248 00000 n
0000002494 00000 n
0000002548 00000 n
0000002602 00000 n
0000002777 00000 n
0000003024 00000 n
0000000016 65535 f
0000000017 65535 f
0000000018 65535 f
0000000019 65535 f
0000000020 65535 f
0000000021 65535 f
0000000022 65535 f
0000000023 65535 f
0000000024 65535 f
0000000025 65535 f
0000000026 65535 f
0000000027 65535 f
0000000028 65535 f
0000000029 65535 f
0000000030 65535 f
0000000031 65535 f
0000000032 65535 f
0000000033 65535 f
0000000034 65535 f
0000000035 65535 f
0000000036 65535 f
0000000037 65535 f
0000000038 65535 f
0000000039 65535 f
0000000040 65535 f
0000000041 65535 f
0000000042 65535 f
0000000043 65535 f
0000000044 65535 f
0000000045 65535 f
0000000046 65535 f
0000000047 65535 f
0000000048 65535 f
0000000049 65535 f
0000000050 65535 f
0000000051 65535 f
0000000052 65535 f
0

Xref와 Trailer 분리

In [61]:
xref_trailer2_only.index("trailer")

2353

In [66]:
xref_only = xref_trailer2_only[:2353]

print(xref_only)

xref
0 117
0000000015 65535 f
0000000017 00000 n
0000000168 00000 n
0000000224 00000 n
0000000508 00000 n
0000001848 00000 n
0000001984 00000 n
0000002012 00000 n
0000002175 00000 n
0000002248 00000 n
0000002494 00000 n
0000002548 00000 n
0000002602 00000 n
0000002777 00000 n
0000003024 00000 n
0000000016 65535 f
0000000017 65535 f
0000000018 65535 f
0000000019 65535 f
0000000020 65535 f
0000000021 65535 f
0000000022 65535 f
0000000023 65535 f
0000000024 65535 f
0000000025 65535 f
0000000026 65535 f
0000000027 65535 f
0000000028 65535 f
0000000029 65535 f
0000000030 65535 f
0000000031 65535 f
0000000032 65535 f
0000000033 65535 f
0000000034 65535 f
0000000035 65535 f
0000000036 65535 f
0000000037 65535 f
0000000038 65535 f
0000000039 65535 f
0000000040 65535 f
0000000041 65535 f
0000000042 65535 f
0000000043 65535 f
0000000044 65535 f
0000000045 65535 f
0000000046 65535 f
0000000047 65535 f
0000000048 65535 f
0000000049 65535 f
0000000050 65535 f
0000000051 65535 f
0000000052 65535 f
0

In [67]:
trailer_only = xref_trailer2_only[2353:]
print(trailer_only)

trailer
<<
/Size 117
/Root 1 0 R
/Info 14 0 R
/ID[<9B8EA23639C1DA4FA67B9B17AEFF023D><9B8EA23639C1DA4FA67B9B17AEFF023D>] >>
startxref
27593
%%EOF



Xref에서 Object가 117개 인 것 확인 가능. 해당 객체들에 접근하기 쉽게 딕셔너리로 저장하기 (Parsing)

In [ ]:
xref_list = xref_only.split('\r\n')

print(xref_list)

['xref', '0 117', '0000000015 65535 f', '0000000017 00000 n', '0000000168 00000 n', '0000000224 00000 n', '0000000508 00000 n', '0000001848 00000 n', '0000001984 00000 n', '0000002012 00000 n', '0000002175 00000 n', '0000002248 00000 n', '0000002494 00000 n', '0000002548 00000 n', '0000002602 00000 n', '0000002777 00000 n', '0000003024 00000 n', '0000000016 65535 f', '0000000017 65535 f', '0000000018 65535 f', '0000000019 65535 f', '0000000020 65535 f', '0000000021 65535 f', '0000000022 65535 f', '0000000023 65535 f', '0000000024 65535 f', '0000000025 65535 f', '0000000026 65535 f', '0000000027 65535 f', '0000000028 65535 f', '0000000029 65535 f', '0000000030 65535 f', '0000000031 65535 f', '0000000032 65535 f', '0000000033 65535 f', '0000000034 65535 f', '0000000035 65535 f', '0000000036 65535 f', '0000000037 65535 f', '0000000038 65535 f', '0000000039 65535 f', '0000000040 65535 f', '0000000041 65535 f', '0000000042 65535 f', '0000000043 65535 f', '0000000044 65535 f', '0000000045 65

In [93]:
# 앞부분에 위치한 `xref`와 `객체 갯수`에 대한 정보와 끝에 존재하는 `빈 문자열` 제거
obj_only = xref_list[2:-1]

In [94]:
obj_dict = {}

i = 0
for obj in obj_only:
    obj_dict[f"obj{i}"] = obj
    i += 1

print(obj_dict)

{'obj0': '0000000015 65535 f', 'obj1': '0000000017 00000 n', 'obj2': '0000000168 00000 n', 'obj3': '0000000224 00000 n', 'obj4': '0000000508 00000 n', 'obj5': '0000001848 00000 n', 'obj6': '0000001984 00000 n', 'obj7': '0000002012 00000 n', 'obj8': '0000002175 00000 n', 'obj9': '0000002248 00000 n', 'obj10': '0000002494 00000 n', 'obj11': '0000002548 00000 n', 'obj12': '0000002602 00000 n', 'obj13': '0000002777 00000 n', 'obj14': '0000003024 00000 n', 'obj15': '0000000016 65535 f', 'obj16': '0000000017 65535 f', 'obj17': '0000000018 65535 f', 'obj18': '0000000019 65535 f', 'obj19': '0000000020 65535 f', 'obj20': '0000000021 65535 f', 'obj21': '0000000022 65535 f', 'obj22': '0000000023 65535 f', 'obj23': '0000000024 65535 f', 'obj24': '0000000025 65535 f', 'obj25': '0000000026 65535 f', 'obj26': '0000000027 65535 f', 'obj27': '0000000028 65535 f', 'obj28': '0000000029 65535 f', 'obj29': '0000000030 65535 f', 'obj30': '0000000031 65535 f', 'obj31': '0000000032 65535 f', 'obj32': '0000000

In [95]:
# 보기 편하게 10개만 출력

i = 0
for key, value in obj_dict.items():
    if i == 10: break
    
    print(f"{key}: {value}")

    i += 1

obj0: 0000000015 65535 f
obj1: 0000000017 00000 n
obj2: 0000000168 00000 n
obj3: 0000000224 00000 n
obj4: 0000000508 00000 n
obj5: 0000001848 00000 n
obj6: 0000001984 00000 n
obj7: 0000002012 00000 n
obj8: 0000002175 00000 n
obj9: 0000002248 00000 n


In [96]:
print(len(obj_dict))

117


---

### **[3] Body**



기본적으로 Body는 간접 객체로 구성되어 있음

**Body 객체 종류**
> Type을 통해 객체가 어떤 종류인지 확인할 수 있음.
> 모든 객체에 Type이 존재하는 것은 아님. 다른 객체를 돕는 객체는 Type이 없을 수 있음.

1. Catalog
2. Pages
3. Page
4. Resource
5. Stream
6. Object Stream

.

.

.

.

---

### **[4] Header**
시작이 되는 부분

예시
```
%PDF-1.7
%도도
```

`%`는 PDF에서 주석을 의미. 
첫 번째 주석은 PDf 버전을 의미
두 번째 주석은 해당 파일이 바이너리 파일이라는 의미 (128 이상의 바이트를 최소 4개 이상 넣음)

### PDF 해석 1단계 (Parsing)

### PDF 해석 2단계 (Decoding)

### PDF 해석 3단계 (Parsing)

### PDF 해석 4단계 (Resource)

### PDF 해석 5단계 (Rendering)